# Numerical Integration and Convergence

## A test problem with a known answer

Numerical integration replaces a continuous area by a finite calculation. To determine whether an algorithm is working, we begin with a function whose exact integral is known:

$$f(x)=\frac{3}{2}(1-x^2), \qquad 0\leq x\leq1.$$

Direct integration gives

$$I=\int_0^1f(x)\,dx
=\frac{3}{2}\left[x-\frac{x^3}{3}\right]_0^1=1.$$

This exact value lets us measure the numerical error

$$E_N=|I_N-I|,$$

where $I_N$ is an estimate constructed with $N$ subintervals or samples. The central question is not just whether an estimate is close to 1, but **how rapidly it approaches 1 as computational effort increases**.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


def integrand(x):
    '''Test function with an exact integral of 1 on [0, 1].'''
    return 1.5*(1.0-x**2)


xlow = 0.0
xhigh = 1.0
exact_integral = 1.0
sample_counts = 10**np.arange(2, 7)  # 10^2 through 10^6


# A dedicated generator makes the experiment exactly reproducible.
rng = np.random.default_rng(441)

## Mean-value Monte Carlo integration

Draw $N$ independent values $X_i$ uniformly from $[a,b]$. Because

$$\mathbb{E}[f(X)]=\frac{1}{b-a}\int_a^b f(x)\,dx,$$

an unbiased Monte Carlo estimator is

$$\hat I_N=(b-a)\frac{1}{N}\sum_{i=1}^{N}f(X_i).$$

The estimator is random: repeating the calculation produces a different answer. Its estimated standard error is

$$\mathrm{SE}(\hat I_N)=(b-a)\frac{s_f}{\sqrt{N}},$$

where $s_f$ is the sample standard deviation of the $f(X_i)$ values. Thus the characteristic statistical error decreases as $N^{-1/2}$, much more slowly than the $N^{-2}$ truncation error of the deterministic rules used in the preceding notebooks.

Smoothness and derivatives do not determine this Monte Carlo convergence rate; the variance of the sampled function values does.

In [ ]:
mc_estimates = []
mc_standard_errors = []

for n_samples in sample_counts:
    x_random = rng.uniform(xlow, xhigh, size=n_samples)
    values = integrand(x_random)
    width = xhigh-xlow
    estimate = width*np.mean(values)
    standard_error = width*np.std(values, ddof=1)/np.sqrt(n_samples)
    mc_estimates.append(estimate)
    mc_standard_errors.append(standard_error)

mc_estimates = np.asarray(mc_estimates)
mc_standard_errors = np.asarray(mc_standard_errors)
mc_absolute_errors = np.abs(mc_estimates-exact_integral)

for n_samples, estimate, standard_error, absolute_error in zip(
        sample_counts, mc_estimates, mc_standard_errors, mc_absolute_errors):
    print(f"N={n_samples:7d}: I_hat={estimate:.8f}, "
          f"SE={standard_error:.3e}, observed |error|={absolute_error:.3e}")

## Statistical uncertainty versus observed error

The exact answer lets us calculate the realized absolute error in this demonstration. In a real Monte Carlo problem the exact answer is normally unknown, so the sample standard error is what quantifies precision.

The two quantities should not be identical. The observed error fluctuates randomly and need not decrease monotonically as $N$ increases. The standard error describes the characteristic scale of those fluctuations over repeated experiments.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].errorbar(sample_counts, mc_estimates, yerr=mc_standard_errors,
                 fmt='o', capsize=4, label=r'Estimate $\pm$ 1 SE')
axes[0].axhline(exact_integral, color='k', linestyle='--', label='Exact integral')
axes[0].set_xscale('log')
axes[0].set_xlabel('Number of random samples, N')
axes[0].set_ylabel('Integral estimate')
axes[0].set_title('Monte Carlo Estimates')
axes[0].legend()

reference = mc_standard_errors[0]*(sample_counts/sample_counts[0])**(-0.5)
axes[1].loglog(sample_counts, mc_absolute_errors, 'o-', label='Observed absolute error')
axes[1].loglog(sample_counts, mc_standard_errors, 's-', label='Estimated standard error')
axes[1].loglog(sample_counts, reference, '--', label=r'Reference: $N^{-1/2}$')
axes[1].set_xlabel('Number of random samples, N')
axes[1].set_ylabel('Error scale')
axes[1].set_title('Monte Carlo Convergence')
axes[1].legend()

slope = np.polyfit(np.log10(sample_counts), np.log10(mc_standard_errors), 1)[0]
print(f"Measured standard-error slope = {slope:.4f} (expected approximately -0.5)")

plt.tight_layout()
plt.show()

## Interpretation

Monte Carlo integration converges slowly in one dimension, so it is not competitive here with midpoint or trapezoid integration. Its advantage appears in high-dimensional problems, where grid-based methods become prohibitively expensive while the characteristic Monte Carlo rate remains $N^{-1/2}$.

Re-run the notebook with different seeds to see how the observed error fluctuates. The standard-error curve should be substantially more stable than any one sequence of realized errors.